In [0]:
df_claims = spark.read.parquet("/Volumes/healthcare_dev/bronze/raw_files/claims/claims.parquet")
df_patients = spark.read.parquet("/Volumes/healthcare_dev/bronze/raw_files/Patients/patients.parquet")
df_providers = spark.read.parquet("/Volumes/healthcare_dev/bronze/raw_files/providers/providers.parquet")
df_encounters = spark.read.parquet("/Volumes/healthcare_dev/bronze/raw_files/encounters/encounters.parquet")
df_observations = spark.read.parquet("/Volumes/healthcare_dev/bronze/raw_files/observations/observations.parquet")
df_conditions = spark.read.parquet("/Volumes/healthcare_dev/bronze/raw_files/conditions/conditions.parquet")

In [0]:
df_claims.columns

['claim_id',
 'patient_id',
 'provider_id',
 'claim_amount',
 'currency',
 'diagnosis_code',
 'claim_status',
 'claim_type',
 'use',
 'event_time',
 'billable_start',
 'billable_end',
 'source_file',
 'is_late_simulated']

In [0]:
late_claim_ids = [
    row["claim_id"]
    for row in
    df_claims
        .filter(
            col("is_late_simulated")==True
        )
        .select("claim_id")
        .collect()
]

In [0]:
len(late_claim_ids)

500

In [0]:
from pyspark.sql.functions import *

claims_df=(
df_claims

.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

.withColumn(
    "batch_id",
    lit("batch_001")
)

.withColumn(
    "source_system",
    lit("FHIR")
)
.withColumn(
    "is_late_simulated",
    when(
        col("claim_id").isin(late_claim_ids),
        True
    )
    .otherwise(False)
)
)

In [0]:
patients_df=(
df_patients

.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

.withColumn(
    "batch_id",
    lit("batch_001")
)

.withColumn(
    "source_system",
    lit("FHIR")
)
)

In [0]:
conditions_df=(
df_conditions

.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

.withColumn(
    "batch_id",
    lit("batch_001")
)

.withColumn(
    "source_system",
    lit("FHIR")
)
)

In [0]:
encounters_df=(
df_encounters

.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

.withColumn(
    "batch_id",
    lit("batch_001")
)

.withColumn(
    "source_system",
    lit("FHIR")
)
)

In [0]:
observations_df=(
df_observations

.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

.withColumn(
    "batch_id",
    lit("batch_001")
)

.withColumn(
    "source_system",
    lit("FHIR")
)
)

In [0]:
providers_df=(
df_providers

.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

.withColumn(
    "batch_id",
    lit("batch_001")
)

.withColumn(
    "source_system",
    lit("FHIR")
)
)

In [0]:
(
    claims_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema","true")
    .saveAsTable("healthcare_dev.bronze.claims")
)

In [0]:
(
    conditions_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema","true")
    .saveAsTable("healthcare_dev.bronze.conditions")
)

In [0]:
(
    patients_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema","true")
    .saveAsTable("healthcare_dev.bronze.patients")
)

In [0]:
(
    encounters_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema","true")
    .saveAsTable("healthcare_dev.bronze.encounters")
)
(
    observations_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema","true")
    .saveAsTable("healthcare_dev.bronze.observations")
)
(
    providers_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema","true")
    .saveAsTable("healthcare_dev.bronze.providers")
)